# 第 3 周练习 — 游戏内容 / 设计数据生成器

## 练习目标（理念）

第 3 周任务：创建**你自己的合成数据生成工具**。输入一种数据类型（产品、职位、游戏内容等），让模型批量产出样本。

本解决方案面向独立 / 独奏开发者的 **游戏内容生成器**：选择内容类型（能力、武器、物品、成就、任务等），补充简短描述或题材，生成可用于原型、内容管线或测试数据的 **JSON 数组**。

## 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions | `client.chat.completions.create(...)` |
| system / user | system 定 schema，user 放题材描述 |
| JSON 输出约束 | `response_format={"type": "json_object"}` |
| 合成数据（Synthetic Data） | 按内容类型批量生成设计条目 |
| Gradio UI | 下拉框 + 滑块 + 一键生成 |

## 怎么跑

1. 准备 `.env`：配置 `OPENROUTER_API_KEY`（本笔记本走 OpenRouter）
2. 从上到下运行单元格，最后 `app.launch()` 打开 Gradio
3. 选内容类型、写描述、调样本数，点 **Generate**


## 为什么这个工具很重要

单人和小团队常常需要**大量不同的内容**（武器、能力、任务、物品），却没有时间手写每一条。生成器以一致的 JSON 形式给出**综合设计数据**，方便你：

- **快速原型化** — 把 JSON 丢进游戏或编辑器，先在系统上迭代。
- **测试 UI 与管线** — 例如库存、战利品表、成就界面。
- **启动内容文档** — 输出当作初稿，再人工润色。

**示例：** 你在做**奇幻 RPG**，需要一批**武器**。内容类型选 **Weapons**，描述写清题材与字段需求，选好样本数后点 **Generate**，就能得到带名称、类型、伤害、描述、稀有度等字段的 JSON 数组，可直接用于游戏数据或设计文档。


In [1]:
# ========== 导入：环境、JSON、正则、OpenAI 客户端、Gradio ==========

# 标准库 os：读环境变量（Environment Variables），例如 API Key
import os
# 标准库 json：解析 / 美化模型返回的 JSON
import json
# 标准库 re：用正则去掉模型可能包上的 Markdown 代码围栏
import re
# load_dotenv：把 .env 里的密钥读进进程环境，避免把密钥写进代码
from dotenv import load_dotenv
# OpenAI 客户端类：这里通过 base_url 指向 OpenRouter 的兼容接口
from openai import OpenAI
# gradio：快速搭 Web UI，让非程序员也能点选生成
import gradio as gr


In [2]:
# ========== 环境与客户端：OpenRouter + 可选模型表 ==========

# override=True：以 .env 覆盖进程里已有同名变量
load_dotenv(override=True)
# 从环境读取 OpenRouter 密钥（字符串名保持原样）
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
# OpenRouter 的 OpenAI 兼容 API 根地址（URL 保持原样）
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# 用兼容端点创建客户端：后续 chat.completions 走 OpenRouter
client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

# 友好名 -> 实际 model id（路由到 OpenRouter / Ollama 等）
MODELS = {
    "gpt-4o-mini": "openai/gpt-4o-mini",
    "Llama": "ollama/llama3.2",
}
# 下拉框默认选中的友好名
DEFAULT_MODEL = "gpt-4o-mini"


In [3]:
# ========== 内容类型字典 + system prompt：用 schema 约束 JSON 字段 ==========

# 每种内容类型：左侧是 UI 标签，右侧是给模型看的字段说明（英文 schema 保持原样，影响输出）
CONTENT_TYPES = {
    "Abilities / skills": "Each object: name, description, effect (what it does), cooldown_or_cost (e.g. '8s' or '20 mana'), rarity (common/rare/epic).",
    "Weapons": "Each object: name, weapon_type (e.g. sword, bow), damage_or_effect, description, rarity.",
    "Items / consumables": "Each object: name, description, effect, consumable (true/false), rarity.",
    "Achievements": "Each object: id, title, description, criteria (how to unlock), reward (optional, e.g. '10 gold').",
    "Quests": "Each object: id, title, description, objectives (array of short strings), reward.",
    "NPCs / characters": "Each object: name, role, one_line, dialogue_hook (short sample line).",
    "Enemies": "Each object: name, type, health_range (e.g. '50-80'), behavior_notes, loot_notes.",
}

# 按内容类型与样本数拼 system prompt（提示词正文保持英文，避免改模型行为）
def get_system_prompt(content_type: str, num_samples: int) -> str:
    # 查表拿 schema；未知类型则给通用字段提示
    schema_hint = CONTENT_TYPES.get(content_type, "Each object: name, description, and any other relevant fields.")
    # f-string：把数量、类型、字段说明注入指令；要求只输出 JSON 对象
    return f"""You are a game design assistant. Generate exactly {num_samples} synthetic game content entries.

Content type: {content_type}
Schema (use these field names): {schema_hint}

Output valid JSON only: a single object with one key "items" (or "data") whose value is an array of {num_samples} objects. No markdown, no explanation. Example shape: {{ "items": [ {{ "name": "...", ... }}, ... ] }}"""


In [4]:
# ========== 解析模型 JSON + 调用 Chat Completions 生成内容 ==========

def extract_json_array(text: str):
    """Get a JSON array from model output (handles wrapper object or raw array)."""
    # 空输出直接报错，方便 UI 侧展示
    if not text or not text.strip():
        raise ValueError("Empty model output.")
    # 去掉首尾空白，便于后续解析
    t = text.strip()
    # 若模型包了 ```json ... ```，用正则剥掉围栏
    if "```" in t:
        t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t, flags=re.IGNORECASE | re.MULTILINE).strip()
    try:
        # 优先整段 json.loads
        obj = json.loads(t)
    except json.JSONDecodeError:
        # 失败则尝试截取第一个 [ 到最后一个 ] 之间的片段
        start, end = t.find("["), t.rfind("]")
        if start != -1 and end != -1 and end > start:
            t = t[start : end + 1]
            # 去掉可能的尾随逗号，再解析
            t = re.sub(r",\s*([\]}])", r"\1", t)  # trailing commas
            obj = json.loads(t)
        else:
            raise ValueError("No JSON array found in output.")
    # 若本身就是 list，直接返回
    if isinstance(obj, list):
        return obj
    # 常见包装键：items / data / entries / results
    for key in ("items", "data", "entries", "results"):
        if isinstance(obj.get(key), list):
            return obj[key]
    raise ValueError("JSON object has no array field (items/data/entries/results).")

def generate_game_content(content_type: str, description: str, num_samples: int, model_name: str) -> str:
    """Generate synthetic game content and return formatted JSON string."""
    # 把样本数夹在 1～20，防止一次请求过大
    num_samples = max(1, min(20, int(num_samples)))
    # system：schema + 数量约束
    system = get_system_prompt(content_type, num_samples)
    # user：有描述用描述，否则用默认题材提示（英文保持原样）
    user = description.strip() if description else "Generate varied, creative entries suitable for a solo or small-team game."
    # 友好名映射到真实 model id；未知则回退默认
    model_id = MODELS.get(model_name, MODELS[DEFAULT_MODEL])
    try:
        # 调用 Chat Completions；要求 json_object，便于稳定解析
        resp = client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            response_format={"type": "json_object"},
        )
        # 取出助手文本；None 时当空串
        raw = resp.choices[0].message.content or ""
        # 从文本抽出数组
        arr = extract_json_array(raw)
        # 美化缩进后返回给 Gradio 展示
        return json.dumps(arr, indent=2)
    except Exception as e:
        # 出错也返回 JSON，UI 仍能显示 error 字段
        return json.dumps({"error": str(e)}, indent=2)


In [5]:
# ========== Gradio UI：下拉内容类型、描述、样本数、模型，一键生成 ==========

# 按钮回调：校验内容类型后转发到 generate_game_content
def run_generate(content_type, description, num_samples, model_name):
    # 未选类型时返回错误 JSON（英文文案保持原样）
    if not content_type:
        return json.dumps({"error": "Pick a content type."}, indent=2)
    return generate_game_content(content_type, description, num_samples, model_name)

# Blocks：组合多个组件的自定义界面
with gr.Blocks(title="Game Content Generator") as app:
    # 界面说明（英文 UI 文案保持原样，避免改产品文案）
    gr.Markdown("Generate synthetic game design data. Choose a content type, add optional description (e.g. genre or theme), and click **Generate**.")
    # 内容类型下拉：选项来自 CONTENT_TYPES 的键
    content_dropdown = gr.Dropdown(
        choices=list(CONTENT_TYPES.keys()),
        value=list(CONTENT_TYPES.keys())[0],
        label="Content type",
    )
    # 可选题材 / 类型描述
    description_box = gr.Textbox(
        label="Description (optional)",
        placeholder="e.g. roguelike abilities, fantasy RPG weapons, sci-fi quests",
        lines=2,
    )
    # 一行里放「样本数滑块」和「模型下拉」
    with gr.Row():
        num_slider = gr.Slider(1, 20, value=5, step=1, label="Number of samples")
        model_dropdown = gr.Dropdown(choices=list(MODELS.keys()), value=DEFAULT_MODEL, label="Model")
    # 主按钮：触发生成
    generate_btn = gr.Button("Generate", variant="primary")
    # 用 Code 组件展示 JSON，带语法高亮
    output_json = gr.Code(label="Generated JSON", language="json", lines=20)

    # 把输入组件接到回调；结果写到 output_json
    generate_btn.click(
        run_generate,
        [content_dropdown, description_box, num_slider, model_dropdown],
        output_json,
    )


In [ ]:
# 启动 Gradio 应用（默认本地端口；浏览器打开后即可交互）
app.launch()
